# ORBITAL EYE AI – U-Net training (Google Colab)

1. **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells **in order** (click ▶ on each one and wait until it finishes).
3. The checkpoint and results are saved in your Google Drive: `MyDrive/orbital_eye/checkpoints/`.

Dataset: Duke DME OCT (Chiu et al., *Biomed. Opt. Express* 6(4), 2015). Research/education use with citation; redistribution prohibited — keep it in your Drive, never in GitHub.

Only the numbers printed by the final **TEST** evaluation are real results. Do not report any other figures.

In [ ]:
# 1) Check the GPU is on (should print a Tesla T4 or similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Connect Google Drive (a pop-up asks you to allow access)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Download the Duke DME dataset into your Drive (178 MB, only the first time)
import os
DATA = '/content/drive/MyDrive/orbital_eye/datasets/duke_dme'
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f'{DATA}/2015_BOE_Chiu/Subject_10.mat'):
    !wget -q --show-progress -O /content/duke.zip https://people.duke.edu/~sf59/Datasets/2015_BOE_Chiu2.zip
    !unzip -q -o /content/duke.zip -d "$DATA"
!ls -lh "$DATA/2015_BOE_Chiu"

In [ ]:
# 4) Get the project code
%cd /content
!rm -rf orbital_eye_ai && git clone -q https://github.com/anitoru/orbital_eye_ai.git
%cd /content/orbital_eye_ai/orbital_eye_ai
!pip install -q -r training/requirements.txt

In [ ]:
# 5) Check the data: should list 10 subjects with 11 annotated B-scans each
!python training/dataset.py --inspect /content/drive/MyDrive/orbital_eye/datasets/duke_dme/2015_BOE_Chiu

In [ ]:
# 6) Train (~20-40 min on a T4). Keep this browser tab open.
#    Model selection uses the validation subjects; the test subjects are evaluated once at the end.
!python training/train.py \
  --data_dir /content/drive/MyDrive/orbital_eye/datasets/duke_dme/2015_BOE_Chiu \
  --out_dir /content/drive/MyDrive/orbital_eye/checkpoints \
  --epochs 80

In [ ]:
# 7) Look at the loss curves and the test metrics
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/orbital_eye/checkpoints/loss_curves.png'))
import json
m = json.load(open('/content/drive/MyDrive/orbital_eye/checkpoints/test_metrics.json'))['test']
print('TEST mean Dice (layers):', m['mean_dice_layers'])
print('TEST mean IoU (layers): ', m['mean_iou_layers'])
print('TEST boundary error (px):', m['mean_boundary_mae_px'])
print('Dice per class [vitreous, 7 layers, below BM]:', m['dice'])

In [ ]:
# 8) Visual check: prediction vs. expert annotation on one TEST B-scan
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '.'); sys.path.insert(0, 'training')
from dataset import load_split, OCTSegDataset
from orbital.ml.unet import UNet
ck = torch.load('/content/drive/MyDrive/orbital_eye/checkpoints/retinal_unet.pt', map_location='cpu')
c = ck['config']; net = UNet(1, c['num_classes'], c['base'], c['depth'], 0.0); net.load_state_dict(ck['model_state']); net.eval()
test = OCTSegDataset(load_split('/content/drive/MyDrive/orbital_eye/datasets/duke_dme/2015_BOE_Chiu', [9, 10]), c['input_height'], c['input_width'])
x, y = test[5]
with torch.no_grad(): p = net(x[None]).argmax(1)[0].numpy()
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(x[0], cmap='gray'); ax[0].set_title('B-scan')
ax[1].imshow(y, cmap='tab10', vmin=0, vmax=9); ax[1].set_title('Expert annotation')
ax[2].imshow(p, cmap='tab10', vmin=0, vmax=9); ax[2].set_title('U-Net prediction')
[a.axis('off') for a in ax]; plt.show()

## Connect the model to the app
1. In Google Drive open `orbital_eye/checkpoints/` and download **`retinal_unet.pt`**.
2. In Codespaces, drag the file into the `orbital_eye_ai/models/` folder (it is git-ignored — do not commit it).
3. In the Codespaces terminal: `pip install -r requirements-ml.txt`, then restart `python app.py`.
4. The **AI model status** page should show **CONNECTED** with the test metrics. New uploads then use the U-Net.